In [ ]:
#!/usr/bin/env python3
"""
src/framework.py
Core implementation of the Closed-Loop ML-RL-XAI Framework:
  1. Hybrid Variability-Runtime Feature Encoder
  2. Fold-Safe SMOTE + Calibrated ML Surrogate (RF + GBDT)
  3. TreeSHAP Attribution & Rolling Action-Pruning Engine
  4. Constraint-Aware PPO Decision Agent (SAT + XAI Logit Masking)
"""

import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler
import shap


def apply_fold_safe_smote(X_train: np.ndarray, y_train: np.ndarray,
                          ratio: float = 0.5, seed: int = 42) -> tuple[np.ndarray, np.ndarray]:
    """Apply SMOTE strictly inside training folds to prevent test leakage."""
    rng = np.random.default_rng(seed)
    pos_idx = np.where(y_train == 1)[0]
    neg_idx = np.where(y_train == 0)[0]
    if len(pos_idx) < 2 or len(pos_idx) >= int(len(neg_idx) * ratio):
        return X_train, y_train

    n_synthetic = int(len(neg_idx) * ratio) - len(pos_idx)
    X_pos = X_train[pos_idx]
    synthetic = []
    for _ in range(n_synthetic):
        i, j = rng.choice(len(X_pos), size=2, replace=False)
        alpha = rng.random()
        synthetic.append(X_pos[i] + alpha * (X_pos[j] - X_pos[i]))
    X_aug = np.vstack([X_train, np.array(synthetic)])
    y_aug = np.concatenate([y_train, np.ones(n_synthetic, dtype=y_train.dtype)])
    return X_aug, y_aug


def build_hybrid_features(C: np.ndarray, E: np.ndarray, interaction_pairs: list[tuple[int, int]]) -> np.ndarray:
    """
    Construct hybrid state matrix x_t = [c_t || h(c_t) || e_t].
    h(c_t) includes pairwise constraint-linked feature products and active feature count.
    """
    n_samples, n_features = C.shape
    active_ratio = (C.sum(axis=1, keepdims=True) / max(1, n_features)).astype(np.float32)
    if interaction_pairs:
        inter_cols = [
            (C[:, i] * C[:, j]).reshape(-1, 1)
            for i, j in interaction_pairs
            if i < n_features and j < n_features
        ]
        H = np.hstack([active_ratio] + inter_cols) if inter_cols else active_ratio
    else:
        H = active_ratio
    return np.hstack([C.astype(np.float32), H.astype(np.float32), E.astype(np.float32)])


class HybridMLSurrogate:
    """Dual-head ML Surrogate (Defect Classifier + Multi-Output NFP Regressor) with TreeSHAP."""

    def __init__(self, seed: int = 42):
        self.seed = seed
        self.scaler = MinMaxScaler()
        self.clf = RandomForestClassifier(
            n_estimators=120, max_depth=12, min_samples_leaf=3, random_state=seed, n_jobs=-1
        )
        self.reg_lat = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=seed)
        self.reg_mem = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=seed)
        self.reg_eng = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=seed)
        self.explainer = None

    def fit(self, X_train: np.ndarray, y_def: np.ndarray,
            y_lat: np.ndarray, y_mem: np.ndarray, y_eng: np.ndarray) -> None:
        X_scaled = self.scaler.fit_transform(X_train)
        X_bal, y_bal = apply_fold_safe_smote(X_scaled, y_def, ratio=0.5, seed=self.seed)
        self.clf.fit(X_bal, y_bal)
        self.reg_lat.fit(X_scaled, y_lat)
        self.reg_mem.fit(X_scaled, y_mem)
        self.reg_eng.fit(X_scaled, y_eng)
        self.explainer = shap.TreeExplainer(self.clf)

    def predict(self, X: np.ndarray) -> dict[str, np.ndarray]:
        X_scaled = self.scaler.transform(X)
        prob_def = self.clf.predict_proba(X_scaled)[:, 1]
        return {
            "defect_prob": prob_def,
            "defect_pred": (prob_def >= 0.5).astype(int),
            "latency_ms": self.reg_lat.predict(X_scaled),
            "memory_mb": self.reg_mem.predict(X_scaled),
            "energy_kwh": self.reg_eng.predict(X_scaled),
        }

    def explain_shap(self, X: np.ndarray, n_config_features: int) -> np.ndarray:
        """Return SHAP feature attributions restricted to the n configuration features."""
        X_scaled = self.scaler.transform(X)
        shap_vals = self.explainer.shap_values(X_scaled)
        # Handle binary classifier list/3D output across SHAP versions
        if isinstance(shap_vals, list):
            sv = shap_vals[1]
        elif shap_vals.ndim == 3:
            sv = shap_vals[:, :, 1]
        else:
            sv = shap_vals
        return sv[:, :n_config_features]


class ClosedLoopPPOAgent:
    """
    Lightweight Policy-Gradient / Clipped-PPO Agent with hard SAT & XAI Logit Masking.
    Supports ablation toggles: use_surrogate, use_sat_mask, use_xai_pruning.
    """

    def __init__(self, state_dim: int, n_features: int, lr: float = 0.015, seed: int = 42):
        self.rng = np.random.default_rng(seed)
        self.n_features = n_features
        self.n_actions = n_features + 1
        self.lr = lr
        # Linear-softmax policy parameters W: (state_dim, n_actions)
        self.W = self.rng.normal(0, 0.05, size=(state_dim, self.n_actions))
        self.b = np.zeros(self.n_actions)
        self.baseline_reward = -0.5

    def _action_probs(self, state: np.ndarray, mask: np.ndarray) -> np.ndarray:
        logits = state @ self.W + self.b
        # Hard logit masking: invalid or XAI-pruned actions receive -1e9
        masked_logits = np.where(mask > 0.5, logits, -1e9)
        masked_logits -= np.max(masked_logits)
        exp_l = np.exp(masked_logits) * (mask > 0.5)
        total = np.sum(exp_l)
        if total <= 0:
            probs = np.zeros_like(mask)
            probs[0] = 1.0
            return probs
        return exp_l / total

    def select_action(self, state: np.ndarray, mask: np.ndarray) -> tuple[int, np.ndarray]:
        probs = self._action_probs(state, mask)
        action = int(self.rng.choice(self.n_actions, p=probs))
        return action, probs

    def update(self, trajectory: list[tuple[np.ndarray, int, float, np.ndarray]]) -> None:
        """PPO-clipped advantage policy update over episode trajectory."""
        if not trajectory:
            return
        returns = []
        G = 0.0
        for _, _, r, _ in reversed(trajectory):
            G = r + 0.99 * G
            returns.insert(0, G)
        returns = np.array(returns)
        self.baseline_reward = 0.9 * self.baseline_reward + 0.1 * float(np.mean(returns))
        advantages = returns - self.baseline_reward
        if np.std(advantages) > 1e-6:
            advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)

        for (state, action, _, old_probs), adv in zip(trajectory, advantages):
            mask = (old_probs > 0).astype(np.float32)
            probs = self._action_probs(state, mask)
            ratio = probs[action] / max(old_probs[action], 1e-8)
            clipped_ratio = np.clip(ratio, 0.8, 1.2)
            effective_adv = min(ratio * adv, clipped_ratio * adv) if adv >= 0 else max(ratio * adv, clipped_ratio * adv)

            grad_logp = -probs
            grad_logp[action] += 1.0
            self.W += self.lr * effective_adv * np.outer(state, grad_logp)
            self.b += self.lr * effective_adv * grad_logp